# Homework 1 — Data Preparation (4 Points)

**Dataset:** [UC Merced Land Use](https://huggingface.co/datasets/blanchon/UC_Merced) — 21 land use classes from aerial imagery.

**Tasks:**
1. Load the dataset from HuggingFace
2. Split into **70% Training / 15% Validation / 15% Test** (stratified)
3. Save to disk in ImageFolder format
4. Show **1 sample image per class**
5. Print the **number of samples per class per split**

## 1. Load the UC Merced Dataset

In [ ]:
from datasets import load_dataset

UC_Merced = load_dataset("blanchon/UC_Merced")

print(UC_Merced)
print(f"\nClass names: {UC_Merced['train'].features['label'].names}")
print(f"Number of classes: {UC_Merced['train'].features['label'].num_classes}")

## 2. Split and Save to Disk

In [ ]:
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

TARGET_BASE = Path("PrepData")

# Get class names
class_names = UC_Merced["train"].features["label"].names

# Combine all HuggingFace splits into one pool
all_images = []
all_labels = []
for split_name in UC_Merced:
    split_data = UC_Merced[split_name]
    all_images.extend(split_data["image"])
    all_labels.extend(split_data["label"])

print(f"Total samples: {len(all_images)}")

# Split into 70% train, 15% val, 15% test (stratified)
train_imgs, temp_imgs, train_labels, temp_labels = train_test_split(
    all_images, all_labels, test_size=0.3, stratify=all_labels, random_state=42
)
val_imgs, test_imgs, val_labels, test_labels = train_test_split(
    temp_imgs, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

In [ ]:
# Save images to disk in ImageFolder format (provided — just run this cell)
splits = {
    "Training": (train_imgs, train_labels),
    "Validation": (val_imgs, val_labels),
    "Test": (test_imgs, test_labels),
}

if TARGET_BASE.exists():
    shutil.rmtree(TARGET_BASE)

for split_name, (images, labels) in splits.items():
    for idx, (img, label) in enumerate(zip(images, labels)):
        cls_name = class_names[label]
        dest_dir = TARGET_BASE / split_name / cls_name
        dest_dir.mkdir(parents=True, exist_ok=True)
        img.convert("RGB").save(dest_dir / f"{idx:04d}.jpg")
    print(f"  {split_name}: {len(images)} images")

print("\nDone! Dataset saved to PrepData/")

## 3. Show 1 Sample Image per Class

In [ ]:
import matplotlib.pyplot as plt
from torchvision import datasets

train_ds = datasets.ImageFolder("PrepData/Training")

fig, axes = plt.subplots(3, 7, figsize=(21, 9))
axes = axes.flatten()

for cls_idx, cls_name in enumerate(train_ds.classes):
    # Find first image of this class
    sample_idx = train_ds.targets.index(cls_idx)
    img, _ = train_ds[sample_idx]

    axes[cls_idx].imshow(img)
    axes[cls_idx].set_title(cls_name, fontsize=9)
    axes[cls_idx].axis('off')

plt.suptitle("1 Sample per Class", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Print Number of Samples per Class per Split

In [ ]:
from collections import Counter

for split_name in ["Training", "Validation", "Test"]:
    ds = datasets.ImageFolder(f"PrepData/{split_name}")
    counts = Counter(ds.targets)

    print(f"\n=== {split_name} ({len(ds)} total) ===")
    for idx in sorted(counts.keys()):
        print(f"  {ds.classes[idx]:30s} {counts[idx]:4d}")